In [1]:
!pip install langchain-groq langchain-community faiss-cpu pypdf sentence-transformers

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Step 1 - Import Libraries

from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("✅ All libraries ready!")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_11304\2063965158.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ All libraries ready!


In [3]:
# Step 2 - Load PDF Resume

import os
print(os.getcwd())

C:\Users\ASUS\resume_analyzer Project


In [5]:
import os
files = os.listdir()
print(files)

['.ipynb_checkpoints', 'ML(MCA).pdf', 'resume_analyzer.ipynb']


In [6]:
# Step 3 - Create Text Chunks

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF load karo
loader = PyPDFLoader("ML(MCA).pdf")
pages = loader.load()
print(f"✅ Pages loaded: {len(pages)}")

# Chunks banao
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(pages)
print(f"✅ Chunks created: {len(chunks)}")

✅ Pages loaded: 1
✅ Chunks created: 5


In [7]:
# Step 4 - Create Embeddings and Vector Store

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Embeddings banao
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Vector store banao
vectorstore = FAISS.from_documents(chunks, embeddings)

print("✅ Vector store ready!")
print(f"✅ Total vectors: {vectorstore.index.ntotal}")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_11304\3875755609.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Vector store ready!
✅ Total vectors: 5


In [8]:
# Step 5 - Connect Groq LLM

from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key="gsk_**********************88888",
    model_name="llama-3.3-70b-versatile"
)

print("✅ LLM ready!")

✅ LLM ready!


In [9]:
# Step 6 - Analyze Resume against Job Description

job_description = """
We are seeking a skilled AI/ML Engineer to design, 
develop, and deploy machine learning models and AI solutions.
Required skills: PyTorch, TensorFlow, Scikit-learn,
NLP, Deep Learning, Generative AI, RAG, FastAPI, Flask.
"""

# Resume se relevant info nikalo
def analyze_resume(job_desc):
    docs = vectorstore.as_retriever().invoke(job_desc)
    resume_context = "\n".join([doc.page_content for doc in docs])
    
    response = llm.invoke(f"""
You are an expert HR analyst.

Analyze the resume against the job description and provide:
1. Match Percentage (out of 100)
2. Strong Skills found in resume
3. Missing Skills not in resume
4. Overall Suggestion

Resume Content:
{resume_context}

Job Description:
{job_desc}

Give clear and structured answer.
""")
    return response.content

# Run karo
result = analyze_resume(job_description)
print(result)

**Analysis of Resume against Job Description**

### 1. Match Percentage (out of 100)
Based on the skills and experience mentioned in the resume and the job description, I would rate the match percentage as **72%**. The candidate has a strong foundation in machine learning and deep learning, with experience in PyTorch, TensorFlow, and Scikit-learn, which are required skills for the job.

### 2. Strong Skills found in resume
The following skills mentioned in the resume are relevant to the job description:
* PyTorch
* TensorFlow
* Scikit-learn
* Python
* Machine Learning
* Deep Learning
* SQL
* Git
* AWS
* Jupyter

### 3. Missing Skills not in resume
The following skills mentioned in the job description are not found in the resume:
* NLP (although the candidate mentions experience in NLP in the summary, it is not explicitly mentioned in the technical skills section)
* Generative AI
* RAG
* FastAPI
* Flask

### 4. Overall Suggestion
While the candidate has a strong foundation in machine le